In [18]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [19]:
df = pd.read_csv("best_data.csv")

In [20]:
df = df.groupby("Percent_Bleaching", group_keys=False).sample(n=1, random_state=42)

In [21]:
df.shape

(1888, 51)

In [22]:
X = df.drop("Percent_Bleaching", axis=1)
Y = df["Percent_Bleaching"]

In [23]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [24]:
hgb_model = HistGradientBoostingRegressor(
    max_iter=394,
    max_depth=15,
    learning_rate=0.0829,
    min_samples_leaf=16,
    l2_regularization=0.094,
    random_state=42
)
# {'max_iter': 394, 'max_depth': 15, 'learning_rate': 0.08291466086400667, 'min_samples_leaf': 16, 'l2_regularization': 0.09425830628658521}

et_model = ExtraTreesRegressor(
    n_estimators=246,
    max_depth=34,
    min_samples_leaf=1,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42
)

# {'n_estimators': 246, 'max_depth': 34, 'min_samples_leaf': 1, 'max_features': 'sqrt'}

rf_model = RandomForestRegressor(
    n_estimators=165, 
    max_depth=28, 
    min_samples_split =  2,
    min_samples_leaf=1,
    max_features='log2',
    n_jobs=-1, 
    random_state=42
)
# {'n_estimators': 165, 'max_depth': 28, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}

voting = VotingRegressor([
    ('hgb', hgb_model),
    ('et',  et_model),
    ('rf',  rf_model)
])


In [25]:
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()
pca = PCA(n_components=0.75)

In [26]:
from sklearn.model_selection import KFold
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ─── ตั้งค่า ───────────────────────────────────────────────
N_SPLITS = 4

# น้ำหนักคงที่จาก robustness analysis
WEIGHTS = {
    'HGB': 0.3160,
    'RF':  0.3129,
    'ET':  0.3711,
}

FEATURES = ['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']
MONTH_COLS = ['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
              'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12']

# X = ข้อมูลดิบทั้งหมดก่อน split (จาก cell 6)
# Y = target (จาก cell 6)


def build_pipeline_one_fold(X_train_full, X_val_full):
    """Refit ทุก preprocessing ใน fold — paper features เข้า PCA เหมือน pipeline จริง"""
    # ตัดแค่ month (paper เข้า PCA เหมือน Cell 8-9)
    X_tr_num    = X_train_full.drop(columns=MONTH_COLS)
    X_val_num   = X_val_full.drop(columns=MONTH_COLS)
    X_tr_month  = X_train_full[MONTH_COLS].copy()
    X_val_month = X_val_full[MONTH_COLS].copy()
    X_tr_fe     = X_train_full[FEATURES].copy()    # paper สำหรับ reattach
    X_val_fe    = X_val_full[FEATURES].copy()

    # Imputer หลัก (ใส่ paper ด้วย เพราะ paper อยู่ใน X_tr_num)
    fold_imp = SimpleImputer(strategy='median')
    X_tr_imp  = fold_imp.fit_transform(X_tr_num)
    X_val_imp = fold_imp.transform(X_val_num)

    # Imputer แยกสำหรับ paper block ที่ reattach (เหมือน fe_imputer ใน Cell 9)
    fold_fe_imp = SimpleImputer(strategy='median')
    X_tr_fe = pd.DataFrame(fold_fe_imp.fit_transform(X_tr_fe),
                           columns=X_tr_fe.columns, index=X_tr_fe.index)
    X_val_fe = pd.DataFrame(fold_fe_imp.transform(X_val_fe),
                            columns=X_val_fe.columns, index=X_val_fe.index)

    fold_scaler = StandardScaler()
    X_tr_ss  = fold_scaler.fit_transform(X_tr_imp)
    X_val_ss = fold_scaler.transform(X_val_imp)

    fold_pca = PCA(n_components=0.75)
    X_tr_pc  = fold_pca.fit_transform(X_tr_ss)
    X_val_pc = fold_pca.transform(X_val_ss)

    pc_cols = [f'PC{i+1}' for i in range(X_tr_pc.shape[1])]
    X_tr_final = pd.concat([
        pd.DataFrame(X_tr_pc, columns=pc_cols),
        X_tr_month.reset_index(drop=True),
        X_tr_fe.reset_index(drop=True),
    ], axis=1)
    X_val_final = pd.concat([
        pd.DataFrame(X_val_pc, columns=pc_cols),
        X_val_month.reset_index(drop=True),
        X_val_fe.reset_index(drop=True),
    ], axis=1)

    return X_tr_final, X_val_final

# ─── Templates ของ model — clone ทุก fold เพื่อให้ fit สดใหม่ ──
hgb_template = HistGradientBoostingRegressor(
    max_iter=394, max_depth=15, learning_rate=0.0829,
    min_samples_leaf=16, l2_regularization=0.094, random_state=42)
et_template = ExtraTreesRegressor(
    n_estimators=246, max_depth=34, min_samples_leaf=1,
    max_features='sqrt', n_jobs=-1, random_state=42)
rf_template = RandomForestRegressor(
    n_estimators=165, max_depth=28, min_samples_split=2,
    min_samples_leaf=1, max_features='log2', n_jobs=-1, random_state=42)


# ─── เริ่ม K-Fold ────────────────────────────────────────────
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
fold_results = []

for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(X), start=1):

    X_tr_full, X_val_full = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val           = Y.iloc[tr_idx], Y.iloc[val_idx]

    # 1. Build pipeline ของ fold นี้ (refit imputer/scaler/pca)
    X_tr_final, X_val_final = build_pipeline_one_fold(X_tr_full, X_val_full)

    # 2. Train model สดใหม่บน training fold
    hgb_f = clone(hgb_template).fit(X_tr_final, y_tr)
    et_f  = clone(et_template ).fit(X_tr_final, y_tr)
    rf_f  = clone(rf_template ).fit(X_tr_final, y_tr)

    # 3. Predict บน validation fold
    pred_hgb = hgb_f.predict(X_val_final)
    pred_et  = et_f .predict(X_val_final)
    pred_rf  = rf_f .predict(X_val_final)

    # 4. คำนวณ ensemble แบบต่าง ๆ
    pred_voting   = (pred_hgb + pred_et + pred_rf) / 3
    pred_weighted = (WEIGHTS['HGB'] * pred_hgb +
                     WEIGHTS['RF']  * pred_rf  +
                     WEIGHTS['ET']  * pred_et)

    # 5. เก็บ metric
    row = {'fold': fold_idx}
    for name, p in [('HGB', pred_hgb), ('RF', pred_rf), ('ET', pred_et),
                    ('Voting', pred_voting), ('Weighted', pred_weighted)]:
        row[f'{name}_R2']   = r2_score(y_val, p)
        row[f'{name}_RMSE'] = np.sqrt(mean_squared_error(y_val, p))
        row[f'{name}_MAE']  = mean_absolute_error(y_val, p)
    fold_results.append(row)

# ─── สรุปผล ──────────────────────────────────────────────────
df_kf = pd.DataFrame(fold_results)

print("=" * 90)
print("K-Fold results (per fold)")
print("=" * 90)
print(df_kf.to_string(index=False, float_format='{:.4f}'.format))

print("\n" + "=" * 90)
print(f"K-Fold mean ± std (n_splits = {N_SPLITS})")
print("=" * 90)
metrics = ['R2', 'RMSE', 'MAE']
models_eval = ['HGB', 'RF', 'ET', 'Voting', 'Weighted']

print(f"{'Model':<12}" + "".join(f"{m + ' (mean ± std)':>26}" for m in metrics))
print("-" * 90)
for mdl in models_eval:
    row_str = f"{mdl:<12}"
    for met in metrics:
        col = f"{mdl}_{met}"
        row_str += f"  {df_kf[col].mean():.4f} ± {df_kf[col].std():.4f}     "
    print(row_str)
print("=" * 90)

K-Fold results (per fold)
 fold  HGB_R2  HGB_RMSE  HGB_MAE  RF_R2  RF_RMSE  RF_MAE  ET_R2  ET_RMSE  ET_MAE  Voting_R2  Voting_RMSE  Voting_MAE  Weighted_R2  Weighted_RMSE  Weighted_MAE
    1  0.5332   18.6055  13.9751 0.5917  17.4003 13.7868 0.5598  18.0675 13.8036     0.5927      17.3797     13.4097       0.5926        17.3820       13.4102
    2  0.5412   18.4513  13.5482 0.5636  17.9947 13.8847 0.5542  18.1863 13.5040     0.5770      17.7165     13.3308       0.5768        17.7197       13.3306
    3  0.5097   18.4615  13.4236 0.5053  18.5437 14.2234 0.5179  18.3071 13.5992     0.5376      17.9276     13.4383       0.5378        17.9237       13.4318
    4  0.5085   18.9101  13.8465 0.5269  18.5523 14.3713 0.5458  18.1782 13.4546     0.5535      18.0232     13.5417       0.5543        18.0069       13.5186

K-Fold mean ± std (n_splits = 4)
Model                  R2 (mean ± std)         RMSE (mean ± std)          MAE (mean ± std)
------------------------------------------------------